# 第2部 機械学習の基本：角度推定

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Asg1223/rcj-soccer-ml/blob/main/notebooks/02_angle_estimation.ipynb)

このノートブックは、教材のページのコードを順番に並べたものです（`tools/build_notebooks.py` で自動生成）。
上のセルから順に `Shift + Enter` で実行してください。

In [ ]:
# Colab のときだけ、教材リポジトリを取ってきて読み込めるようにする
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !git clone -q --depth 1 https://github.com/Asg1223/rcj-soccer-ml.git /content/rcj-soccer-ml
    sys.path.insert(0, "/content/rcj-soccer-ml")
print("Colab で実行中" if IN_COLAB else "ローカルで実行中")

## 2-1 まずは従来の方法：重み付きベクトル和

説明は教材のページを見てください：https://asg1223.github.io/rcj-soccer-ml/part2-ml-basics/baseline/

#### 1. データを用意する

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rcjml import load_config
from rcjml.data import load_synthetic, sensor_columns, split_by_session

cfg = load_config()
df = load_synthetic("ring12_large_normal")
S = sensor_columns(df)
train, test = split_by_session(df)       # 取り直しの後ろ 1/4 をテスト用にする
print(len(train), len(test))

#### 2. 自分で書く

In [ ]:
def vector_sum_angle(values, sensor_angles_deg):
    phi = np.deg2rad(sensor_angles_deg)         # 度 → ラジアン
    x = (values * np.cos(phi)).sum(axis=1)      # 行ごとに足す
    y = (values * np.sin(phi)).sum(axis=1)
    return np.rad2deg(np.arctan2(y, x))         # ラジアン → 度

pred = vector_sum_angle(test[S].values, cfg.ir.angles_deg)
pred[:5].round(1), test["angle_deg"].values[:5]

#### 3. 誤差を測る

In [ ]:
from rcjml.sim import angle_error_deg

err = angle_error_deg(pred, test["angle_deg"])
print(f"平均誤差（MAE）: {err.mean():.1f}°")
print(f"真ん中の値     : {np.median(err):.1f}°")
print(f"90% の範囲     : {np.percentile(err, 90):.1f}° 以内")

#### 4. どこで外しているか

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].hist(err, bins=60)
axes[0].set_xlabel("angle error [deg]")
axes[0].set_ylabel("count")
axes[0].set_yscale("log")

axes[1].scatter(test["distance_cm"], err, s=4, alpha=0.4)
axes[1].set_xlabel("distance [cm]")
axes[1].set_ylabel("angle error [deg]")
plt.tight_layout()
plt.show()

#### 5. 難しさを変えて比べる

In [ ]:
results = {}
for name in ["ring12_large_easy", "ring12_large_normal", "ring12_large_hard", "ring12_golf_normal"]:
    d = load_synthetic(name)
    _, t = split_by_session(d)
    e = angle_error_deg(vector_sum_angle(t[S].values, cfg.ir.angles_deg), t["angle_deg"])
    results[name] = {"MAE": e.mean(), "median": np.median(e), "p90": np.percentile(e, 90)}
pd.DataFrame(results).T.round(1)

## 2-2 重みをデータから決める：線形回帰と勾配降下法

説明は教材のページを見てください：https://asg1223.github.io/rcj-soccer-ml/part2-ml-basics/linear-regression/

### 手を動かす 1：重みが1つだけのモデル

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from rcjml import load_config
from rcjml.data import load_synthetic, sensor_columns, split_by_session
from rcjml.sim import angle_error_deg

cfg = load_config()
df = load_synthetic("ring12_large_normal")
S = sensor_columns(df)
train, test = split_by_session(df)

x = (train["s0"] - train["s6"]).values / cfg.ir.max_reading   # 正面 − 後ろ（0〜1 にそろえる）
y = np.cos(np.deg2rad(train["angle_deg"].values))             # 正面なら 1、後ろなら -1

w = 0.0          # 最初の重み
lr = 1.0         # 学習率
history = []
for step in range(30):
    pred = w * x
    loss = ((pred - y) ** 2).mean()
    grad = (2 * (pred - y) * x).mean()     # 傾き
    history.append((step, w, loss))
    w = w - lr * grad                      # 坂を下る

for step, w_, loss in history[:6] + history[-1:]:
    print(f"{step:2d} 回目  w = {w_:.3f}  損失 = {loss:.4f}")

### 手を動かす 2：12 個のセンサ全部を使う

In [ ]:
def features(d):
    X = d[S].values / cfg.ir.max_reading            # 0〜1 にそろえる
    return np.c_[X, np.ones(len(d))]                 # 最後にバイアス用の 1 の列を足す

def targets(d):
    th = np.deg2rad(d["angle_deg"].values)
    return np.c_[np.cos(th), np.sin(th)]

X, Y = features(train), targets(train)
W = np.zeros((X.shape[1], 2))      # 重み（最初は全部 0）
lr = 0.2
losses = []
for step in range(2000):
    P = X @ W                                  # 全データの予測
    grad = 2 * X.T @ (P - Y) / len(X)          # 傾き（全部の重みの分をまとめて）
    W = W - lr * grad
    losses.append(((P - Y) ** 2).mean())

plt.plot(losses)
plt.xlabel("step")
plt.ylabel("loss (MSE)")
plt.yscale("log")
plt.show()

### 手を動かす 3：テスト用データで角度を出す

In [ ]:
P = features(test) @ W
pred_angle = np.rad2deg(np.arctan2(P[:, 1], P[:, 0]))
err = angle_error_deg(pred_angle, test["angle_deg"])
print(f"線形回帰の平均誤差: {err.mean():.1f}°")

### 手を動かす 4：学んだ重みをのぞく

In [ ]:
learned_dir = np.rad2deg(np.arctan2(W[:-1, 1], W[:-1, 0]))
design_dir = (np.array(cfg.ir.angles_deg) + 180) % 360 - 180
table = np.c_[design_dir, learned_dir.round(0)]
for i, (d, l) in enumerate(table):
    print(f"s{i:<2d} 設計 {d:6.0f}°   学んだ向き {l:6.0f}°")

## 2-3 角度は輪っか：359° と 1° の問題

説明は教材のページを見てください：https://asg1223.github.io/rcj-soccer-ml/part2-ml-basics/angle-ring/

### 手を動かす 1：角度をそのまま予測してみる（わざと失敗する）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from rcjml import load_config
from rcjml.data import load_synthetic, sensor_columns, split_by_session

cfg = load_config()
df = load_synthetic("ring12_large_normal")
S = sensor_columns(df)
train, test = split_by_session(df)

def features(d):
    return np.c_[d[S].values / cfg.ir.max_reading, np.ones(len(d))]

w, *_ = np.linalg.lstsq(features(train), train["angle_deg"].values, rcond=None)
pred_raw = features(test) @ w

### 手を動かす 2：誤差の関数を自分で書く

In [ ]:
def angle_error(pred_deg, true_deg):
    diff = (np.asarray(pred_deg) - np.asarray(true_deg) + 180) % 360 - 180
    return np.abs(diff)

print(angle_error(179, -179))    # 2 になれば正解
print(angle_error(10, 350))      # 20 になれば正解

### 手を動かす 3：比べる

In [ ]:
th = np.deg2rad(train["angle_deg"].values)
W, *_ = np.linalg.lstsq(features(train), np.c_[np.cos(th), np.sin(th)], rcond=None)
P = features(test) @ W
pred_ring = np.rad2deg(np.arctan2(P[:, 1], P[:, 0]))

err_raw = angle_error(pred_raw, test["angle_deg"])
err_ring = angle_error(pred_ring, test["angle_deg"])
print(f"角度をそのまま : {err_raw.mean():.1f}°")
print(f"cos と sin     : {err_ring.mean():.1f}°")

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, pred, name in [(axes[0], pred_raw, "raw angle"), (axes[1], pred_ring, "cos / sin")]:
    ax.scatter(test["angle_deg"], pred, s=3, alpha=0.4)
    ax.plot([-180, 180], [-180, 180], color="tab:orange")
    ax.set_xlabel("true angle [deg]")
    ax.set_title(name)
axes[0].set_ylabel("predicted angle [deg]")
plt.show()

## 2-4 正しく評価する：学習用とテスト用の分け方

説明は教材のページを見てください：https://asg1223.github.io/rcj-soccer-ml/part2-ml-basics/evaluation/

### 手を動かす：分け方で成績がどれだけ変わるか

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from rcjml import load_config
from rcjml.data import load_synthetic, sensor_columns, split_by_session
from rcjml.sim import angle_error_deg

cfg = load_config()
df = load_synthetic("ring12_large_hard")
S = sensor_columns(df)

def xy(d):
    th = np.deg2rad(d["angle_deg"].values)
    return d[S].values / cfg.ir.max_reading, np.c_[np.cos(th), np.sin(th)]

def score(model, train, test):
    X, Y = xy(train)
    model.fit(X, Y)
    P = model.predict(xy(test)[0])
    pred = np.rad2deg(np.arctan2(P[:, 1], P[:, 0]))
    return angle_error_deg(pred, test["angle_deg"]).mean()

splits = {
    "random": train_test_split(df, test_size=0.25, random_state=0),
    "by session": split_by_session(df),
}
models = {
    "KNN (k=5)": lambda: KNeighborsRegressor(n_neighbors=5),
    "random forest": lambda: RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=0),
}
rows = {m: {s: score(make(), *splits[s]) for s in splits} for m, make in models.items()}
pd.DataFrame(rows).T.round(1)

## 2-5 scikit-learn で数行に

説明は教材のページを見てください：https://asg1223.github.io/rcj-soccer-ml/part2-ml-basics/scikit-learn/

### 手を動かす 1：線形回帰を scikit-learn で

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from rcjml import load_config
from rcjml.data import load_synthetic, sensor_columns, split_by_session
from rcjml.sim import angle_error_deg

cfg = load_config()
df = load_synthetic("ring12_large_normal")
S = sensor_columns(df)
train, test = split_by_session(df)

def xy(d):
    th = np.deg2rad(d["angle_deg"].values)
    return d[S].values / cfg.ir.max_reading, np.c_[np.cos(th), np.sin(th)]

def to_angle(P):
    return np.rad2deg(np.arctan2(P[:, 1], P[:, 0]))

X_train, Y_train = xy(train)
X_test, _ = xy(test)

model = LinearRegression()
model.fit(X_train, Y_train)
err = angle_error_deg(to_angle(model.predict(X_test)), test["angle_deg"])
print(f"LinearRegression: {err.mean():.1f}°")

### 手を動かす 2：モデルを取りかえて比べる

In [ ]:
import time
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from rcjml.sim import baseline_angle

models = {
    "linear": LinearRegression(),
    "KNN (k=5)": KNeighborsRegressor(n_neighbors=5),
    "random forest": RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=0),
}

results = {"vector sum (2-1)": angle_error_deg(baseline_angle(cfg, test[S].values), test["angle_deg"]).mean()}
for name, model in models.items():
    t0 = time.time()
    model.fit(X_train, Y_train)
    results[name] = angle_error_deg(to_angle(model.predict(X_test)), test["angle_deg"]).mean()
    print(f"{name:14s} 学習 {time.time() - t0:.1f} 秒")

pd.Series(results).round(1)

### 手を動かす 3：どこが良くなったか見る

In [ ]:
import matplotlib.pyplot as plt

rf = models["random forest"]
err_rf = angle_error_deg(to_angle(rf.predict(X_test)), test["angle_deg"])
err_base = angle_error_deg(baseline_angle(cfg, test[S].values), test["angle_deg"])

bins = pd.cut(test["distance_cm"], [0, 20, 40, 80, 120, 180])
table = pd.DataFrame({"vector sum": err_base, "random forest": err_rf}).groupby(bins.values, observed=True).mean()
table.round(1).plot.bar(figsize=(7, 3.5), rot=0, ylabel="mean angle error [deg]")
plt.show()

### でも、このモデルはロボットに載るか？

In [ ]:
import pickle
for name, model in models.items():
    print(f"{name:14s} 大きさ {len(pickle.dumps(model)) / 1e6:8.2f} MB")